# Pandas Basics Recap

Before we dive into real-world data wrangling with the MovieLens dataset, let's recap the core building blocks of pandas.

In this notebook, we will cover:

- What a pandas **Series** is, how to create one, and why we need it
- Moving from a **Series** to a **DataFrame**
- The difference between a **Series** and a **DataFrame**

## Pandas Series

A **Series** is a one-dimensional array with labels attached to each value.

Think of a plain Python list of exam scores:

In [2]:
scores = [85, 90, 78, 92]

If you want to know whose score is 90, you'd have to remember "that was the 2nd item." There's no name attached to each value, just position numbers.

A Series fixes this by attaching a label (called an **index**) to every value.

In [3]:
import pandas as pd

scores = pd.Series([85, 90, 78, 92], index=['Ria', 'Beth', 'Carl', 'Dana'])
scores

Ria     85
Beth    90
Carl    78
Dana    92
dtype: int64

### Why do we need a Series?

- Real-world data almost never comes as a bare list of numbers, it comes with something that identifies each row (a user ID, a date, a movie title).
- A Series lets us look up values by a meaningful label, instead of remembering position numbers.
- It's still built on top of a NumPy array, so all the fast, vectorized operations you already know still work the same way.

In [4]:
scores['Beth']

np.int64(90)

In [5]:
scores.values

array([85, 90, 78, 92])

## From Series to DataFrame

A Series is great for one column of labeled data. But real datasets have many columns, age, gender, rating, occupation, and so on, all lined up under the same row labels.

A **DataFrame** is what you get when you put multiple Series side by side, sharing the same index.

Think of it like a spreadsheet:
- Each **column** in the spreadsheet is a Series
- The **whole spreadsheet** (all columns together) is a DataFrame

In [6]:
ages = pd.Series([25, 30, 22, 28], index=['Ria', 'Beth', 'Carl', 'Dana'])
scores = pd.Series([85, 90, 78, 92], index=['Ria', 'Beth', 'Carl', 'Dana'])

If we combine these two Series into one table, using a dictionary where each key becomes a column name:

In [7]:
df = pd.DataFrame({'age': ages, 'score': scores})
df

,age,score
Ria,25,85
Beth,30,90
Carl,22,78
Dana,28,92


In [8]:
# Pulling a single column back out gives you a Series again
df['age']

Ria     25
Beth    30
Carl    22
Dana    28
Name: age, dtype: int64

In [9]:
type(df['age'])

pandas.core.series.Series


- A **DataFrame is a collection of Series**, all lined up on the same index.
- Pulling one column out of a DataFrame (`df['age']`) gives you back a Series.
- Putting several Series together (`pd.DataFrame({...})`) gives you a DataFrame.

## Exploring a DataFrame

Now that we know what a Series and DataFrame are, let's practice the everyday methods you'll use on any DataFrame, using a small example table before we move to the real dataset.

In this section, we will cover:

- Basic inspection: `.head()`, `.tail()`, `.index`, `.columns`, `.dtypes`, `.shape`
- Descriptive statistics: `.describe()`, `.mean()`, `.min()`, `.max()`, `.std()`, `.mode()`, `.corr()`
- Selecting rows: `.loc` vs `.iloc`
- Filtering rows with conditions
- Handling missing data: `.isnull()`, `.dropna()`
- Data visualization: histogram, boxplot
- Data manipulation: slicing, groupby/aggregate, merging, string operations, timestamps, sorting

## Basic Inspection Methods

Before doing any analysis, it's good practice to get some details about the dataframe, its size, its columns, and what a few rows actually look like.

Let's create a small example DataFrame to practice on:

In [10]:
data = {
    'fruit': ['Apple', 'Banana', 'Mango', 'Grape', 'Orange', 'Pineapple', 'Watermelon',
              'Strawberry', 'Blueberry', 'Papaya', 'Kiwi', 'Peach', 'Pear', 'Cherry', 'Plum'],
    'price_thb': [55, 15, 40, 90, 35, 45, 60, 120, 150, 30, 65, 80, 70, 180, 60],
    'quantity_sold': [120, 200, 90, 60, 150, 40, 35, 75, 55, 100, 65, 110, 95, 30, 85]
}

df = pd.DataFrame(data)
df

,fruit,price_thb,quantity_sold
0,Apple,55,120
1,Banana,15,200
2,Mango,40,90
3,Grape,90,60
4,Orange,35,150
5,Pineapple,45,40
6,Watermelon,60,35
7,Strawberry,120,75
8,Blueberry,150,55
9,Papaya,30,100


### Looking at rows: `.head()` and `.tail()`

- `.head()` shows the first 5 rows by default (you can pass a number for more/fewer)
- `.tail()` shows the last 5 rows

In [11]:
df.head()


,fruit,price_thb,quantity_sold
0,Apple,55,120
1,Banana,15,200
2,Mango,40,90
3,Grape,90,60
4,Orange,35,150


In [12]:
df.tail()

,fruit,price_thb,quantity_sold
10,Kiwi,65,65
11,Peach,80,110
12,Pear,70,95
13,Cherry,180,30
14,Plum,60,85


In [13]:
df.head(3)

,fruit,price_thb,quantity_sold
0,Apple,55,120
1,Banana,15,200
2,Mango,40,90


In [14]:
df.tail(2)

,fruit,price_thb,quantity_sold
13,Cherry,180,30
14,Plum,60,85


### Looking at the structure: `.index`, `.columns`, `.dtypes`, `.shape`

- `.index`: the row labels
- `.columns`: the column names
- `.dtypes`: the data type of each column
- `.shape`: (number of rows, number of columns)

In [15]:
df.index

RangeIndex(start=0, stop=15, step=1)

In [16]:
df.columns

Index(['fruit', 'price_thb', 'quantity_sold'], dtype='object')

In [17]:
df.dtypes

fruit            object
price_thb         int64
quantity_sold     int64
dtype: object

In [18]:
df.shape

(15, 3)

## Descriptive Statistics

Once we know the shape and structure of a DataFrame, the next step is to understand what the data actually looks like.

In [19]:
df.describe()

,price_thb,quantity_sold
count,15.0000,15.000000
mean,73.0000,87.333333
std,45.6618,45.625285
min,15.0000,30.000000
25%,42.5000,57.500000
50%,60.0000,85.000000
75%,85.0000,105.000000
max,180.0000,200.000000


`.describe()` gives a quick summary for numeric columns:
- `count` : number of non-missing values
- `mean` : average value
- `std` : standard deviation (how spread out the values are)
- `min` / `max` : smallest and largest values
- `25%`, `50%`, `75%` : quartiles (50% is the median)

We can also pull out individual statistics for a single column:

In [20]:
df['price_thb'].mean()

np.float64(73.0)

In [21]:
df['price_thb'].min()

np.int64(15)

In [22]:
df['price_thb'].max()

np.int64(180)

In [23]:
df['price_thb'].std()

np.float64(45.661800227323496)

In [24]:
df['price_thb'].mode()

0    60
Name: price_thb, dtype: int64

### Correlation

`.corr()` tells us how strongly two numeric columns move together. Values range from -1 to 1:
- Close to **1** → strong positive relationship (both go up together)
- Close to **-1** → strong negative relationship (one goes up, the other goes down)
- Close to **0** → little to no relationship

In [25]:
df.corr(numeric_only=True)

,price_thb,quantity_sold
price_thb,1.000000,-0.598457
quantity_sold,-0.598457,1.000000


### Class Exercise: Descriptive Statistics (5 points)

Using the `df` DataFrame above, answer the following:

- Get the summary statistics for the `quantity_sold` column using `.describe()`
  
- What is the average `quantity_sold`?
  
- What is the maximum `price_thb`?
  
- Is there a strong correlation between `price_thb` and `quantity_sold`? What does that tell you? (1+1 points)

In [26]:
#Get the summary statistics for the `quantity_sold` column using `.describe()`

#YOUR_CODE_HERE

In [27]:
#What is the average `quantity_sold`?

#YOUR_CODE_HERE

In [28]:
#What is the maximum `price_thb`?

#YOUR_CODE_HERE

In [29]:
#Is there a strong correlation between `price_thb` and `quantity_sold`? What does that tell you?

#YOUR_CODE_HERE

## Selecting Rows: `.loc` vs `.iloc`

There are two main ways to select rows from a DataFrame:

- **`.loc`** : Selects by **label** (the actual index value, or column name)
- **`.iloc`** : Selects by **position** (like a regular Python list, starting at 0)

In [30]:
df.head()

,fruit,price_thb,quantity_sold
0,Apple,55,120
1,Banana,15,200
2,Mango,40,90
3,Grape,90,60
4,Orange,35,150


With the default index (0, 1, 2, ...), `.loc[0]` and `.iloc[0]` look the same. The difference shows up clearly once the index is no longer just plain numbers.

In [31]:
df_indexed = df.set_index('fruit')
df_indexed

,price_thb,quantity_sold
fruit,,
Apple,55,120
Banana,15,200
Mango,40,90
Grape,90,60
Orange,35,150
Pineapple,45,40
Watermelon,60,35
Strawberry,120,75
Blueberry,150,55


In [32]:
# .loc uses the label, the fruit name

df_indexed.loc['Mango']

price_thb        40
quantity_sold    90
Name: Mango, dtype: int64

In [33]:
# .iloc uses the position, regardless of what the label is

df_indexed.iloc[2]

price_thb        40
quantity_sold    90
Name: Mango, dtype: int64

Both return the same row here, but for different reasons:
- `.loc['Mango']` : "give me the row labeled Mango"
- `.iloc[2]` : "give me the row at position 2" (0-indexed: Apple, Banana, Mango...)

We can also select multiple rows, or a range:

In [34]:
df_indexed.loc[['Mango', 'Kiwi']]

,price_thb,quantity_sold
fruit,,
Mango,40,90
Kiwi,65,65


In [35]:
df_indexed.iloc[0:3]

,price_thb,quantity_sold
fruit,,
Apple,55,120
Banana,15,200
Mango,40,90


## Filtering Rows with Conditions

Filtering lets us pick out only the rows that meet a certain condition, similar to the masking we did with NumPy arrays in Week 2.

The idea is the same: we build a **boolean mask** (True/False for each row), then use it to select only the rows where the condition is True.

In [36]:
# Boolean mask: True where price is above 60 THB
df['price_thb'] > 60

0     False
1     False
2     False
3      True
4     False
5     False
6     False
7      True
8      True
9     False
10     True
11     True
12     True
13     True
14    False
Name: price_thb, dtype: bool

In [37]:
# Use the mask to filter the DataFrame
df[df['price_thb'] > 60]

,fruit,price_thb,quantity_sold
3,Grape,90,60
7,Strawberry,120,75
8,Blueberry,150,55
10,Kiwi,65,65
11,Peach,80,110
12,Pear,70,95
13,Cherry,180,30


We can also combine multiple conditions using:
- `&` for **and**
- `|` for **or**

Each condition needs to be wrapped in parentheses.

In [38]:
# Fruits priced above 60 THB AND sold more than 50 times
df[(df['price_thb'] > 60) & (df['quantity_sold'] > 50)]

,fruit,price_thb,quantity_sold
3,Grape,90,60
7,Strawberry,120,75
8,Blueberry,150,55
10,Kiwi,65,65
11,Peach,80,110
12,Pear,70,95


In [39]:
# Fruits priced above 100 THB OR sold more than 150 times
df[(df['price_thb'] > 100) | (df['quantity_sold'] > 150)]

,fruit,price_thb,quantity_sold
1,Banana,15,200
7,Strawberry,120,75
8,Blueberry,150,55
13,Cherry,180,30


### Class Exercise: Selecting and Filtering (5 points)

Using the `df_indexed` DataFrame (fruit as index), answer the following:

- Select the row for `'Cherry'` using `.loc`
  
- Select the row at position 5 using `.iloc`
  
- Filter `df` to show only fruits with `quantity_sold` greater than 100
  
- Filter `df` to show fruits priced under 50 THB **and** sold more than 80 times
  

In [40]:
#Select the row for `'Cherry'` using `.loc`

#YOUR_CODE_HERE

In [41]:
#Select the row at position 5 using `.iloc`

#YOUR_CODE_HERE

In [42]:
#Filter `df` to show only fruits with `quantity_sold` greater than 100

#YOUR_CODE_HERE

In [43]:
#Filter `df` to show fruits priced under 50 THB **and** sold more than 80 times

#YOUR_CODE_HERE

## Handling Missing Data

Real datasets almost always have gaps, missing values that weren't recorded. Before doing any analysis, it's important to check for these and decide how to handle them.

Let's introduce some missing values into our fruit DataFrame to practice on:

In [44]:
import numpy as np

df_missing = df.copy()
df_missing.loc[2, 'price_thb'] = np.nan
df_missing.loc[7, 'quantity_sold'] = np.nan
df_missing.loc[10, 'price_thb'] = np.nan

df_missing

,fruit,price_thb,quantity_sold
0,Apple,55.0,120.0
1,Banana,15.0,200.0
2,Mango,NaN,90.0
3,Grape,90.0,60.0
4,Orange,35.0,150.0
5,Pineapple,45.0,40.0
6,Watermelon,60.0,35.0
7,Strawberry,120.0,NaN
8,Blueberry,150.0,55.0
9,Papaya,30.0,100.0


### Checking for missing values: `.isnull()`

`.isnull()` returns True/False for every cell, marking where values are missing.

In [45]:
df_missing.isnull()

,fruit,price_thb,quantity_sold
0,False,False,False
1,False,False,False
2,False,True,False
3,False,False,False
4,False,False,False
5,False,False,False
6,False,False,False
7,False,False,True
8,False,False,False
9,False,False,False


A more useful summary is combining it with `.sum()`, which counts how many `True` values (missing values) are in each column.

In [46]:
df_missing.isnull().sum()

fruit            0
price_thb        2
quantity_sold    1
dtype: int64

### Handling missing values: `.dropna()`

One way to deal with missing values is to simply remove the rows that contain them.

In [47]:
df_cleaned = df_missing.dropna()
df_cleaned

,fruit,price_thb,quantity_sold
0,Apple,55.0,120.0
1,Banana,15.0,200.0
3,Grape,90.0,60.0
4,Orange,35.0,150.0
5,Pineapple,45.0,40.0
6,Watermelon,60.0,35.0
8,Blueberry,150.0,55.0
9,Papaya,30.0,100.0
11,Peach,80.0,110.0
12,Pear,70.0,95.0


In [48]:
# Confirm no more missing values
df_cleaned.isnull().sum()

fruit            0
price_thb        0
quantity_sold    0
dtype: int64

### Class Exercise: Handling Missing Data (5 points)

- Check how many missing values exist in each column of `df_missing`
  
- Instead of dropping rows, try filling the missing `price_thb` values with the column's mean (**HINT:** use `.fillna()`)
  
- Check again if any missing values remain in price_thb column

In [49]:
#Check how many missing values exist in each column of `df_missing`

#YOUR_CODE_HERE

In [50]:
#Make a copy of the dataframe

#YOUR_CODE_HERE

In [51]:
#Check the mean of "price_thb" in df_filled
#Hint use .mean()

#YOUR_CODE_HERE

In [52]:
#Fill the missing `price_thb` values with the column's mean (HINT: use `.fillna()`)

#YOUR_CODE_HERE

In [53]:
#Check again if any missing values remain in price_thb column

#YOUR_CODE_HERE

## Merging DataFrames (Joins)

Now let's understand the different **types of joins** available, since choosing the right one matters a lot for the accuracy of your analysis.

When merging two DataFrames on a shared key, there are six common ways to combine them:

![Join Types](images/df_joins.jpg)

| Join Type | Keeps | Venn Diagram |
|---|---|---|
| Inner | Only rows where the key exists in **both** DataFrames | Overlapping middle |
| Left | **All** rows from the left DataFrame, matched values from the right where available | Left circle, fully filled |
| Right | **All** rows from the right DataFrame, matched values from the left where available | Right circle, fully filled |
| Full Outer | **All** rows from both DataFrames, matched where possible | Both circles, fully filled |
| Left (if NULL) | Only left rows with **no match** in the right | Left circle, excluding overlap |
| Right (if NULL) | Only right rows with **no match** in the left | Right circle, excluding overlap |

In [54]:
# Left DataFrame: students and the course they enrolled in
students = pd.DataFrame({
    'student_id': [1, 2, 3, 4, 5],
    'name': ['Anan', 'Bella', 'Chai', 'Dara', 'Enzo'],
    'course_id': [101, 102, 103, 104, 999]
})
students

,student_id,name,course_id
0,1,Anan,101
1,2,Bella,102
2,3,Chai,103
3,4,Dara,104
4,5,Enzo,999


In [55]:
# Right DataFrame: course details
courses = pd.DataFrame({
    'course_id': [101, 102, 103, 105],
    'course_name': ['Math', 'Physics', 'Chemistry', 'Biology']
})
courses

,course_id,course_name
0,101,Math
1,102,Physics
2,103,Chemistry
3,105,Biology


### Inner Join

Keeps only rows where `course_id` exists in **both** `students` and `courses`.

In [56]:
inner_result = pd.merge(students, courses, on='course_id', how='inner')
inner_result

,student_id,name,course_id,course_name
0,1,Anan,101,Math
1,2,Bella,102,Physics
2,3,Chai,103,Chemistry


### Left Join

Keeps **all** rows from `students` (the left DataFrame), filling in course details where a match exists, and `NaN` where it doesn't.

In [57]:
left_result = pd.merge(students, courses, on='course_id', how='left')
left_result

,student_id,name,course_id,course_name
0,1,Anan,101,Math
1,2,Bella,102,Physics
2,3,Chai,103,Chemistry
3,4,Dara,104,NaN
4,5,Enzo,999,NaN


### Right Join

Keeps **all** rows from `courses` (the right DataFrame), filling in student details where a match exists, and `NaN` where it doesn't.

In [58]:
right_result = pd.merge(students, courses, on='course_id', how='right')
right_result

,student_id,name,course_id,course_name
0,1.0,Anan,101,Math
1,2.0,Bella,102,Physics
2,3.0,Chai,103,Chemistry
3,NaN,NaN,105,Biology


### Full Outer Join

Keeps **all** rows from both DataFrames, matching where possible and filling `NaN` everywhere else.

In [60]:
outer_result = pd.merge(students, courses, on='course_id', how='outer')
outer_result

,student_id,name,course_id,course_name
0,1.0,Anan,101,Math
1,2.0,Bella,102,Physics
2,3.0,Chai,103,Chemistry
3,4.0,Dara,104,NaN
4,NaN,NaN,105,Biology
5,5.0,Enzo,999,NaN


### Left Join (if NULL)

Sometimes we don't want the matched rows at all, we specifically want to find the left rows that have **no match** on the right. Real use case: "which students are enrolled in a course that doesn't exist in our course catalog?"

We do this by performing a left join with `indicator=True`, then filtering to keep only the rows where the match came from the left side only.

In [61]:
left_join_full = pd.merge(students, courses, on='course_id', how='left', indicator=True)
left_join_full

,student_id,name,course_id,course_name,_merge
0,1,Anan,101,Math,both
1,2,Bella,102,Physics,both
2,3,Chai,103,Chemistry,both
3,4,Dara,104,NaN,left_only
4,5,Enzo,999,NaN,left_only


In [62]:
left_only = left_join_full[left_join_full['_merge'] == 'left_only']
left_only

,student_id,name,course_id,course_name,_merge
3,4,Dara,104,NaN,left_only
4,5,Enzo,999,NaN,left_only


### Right Join (if NULL)

Same idea, but the other direction, right rows with no match on the left. Real use case: "which courses have no students enrolled?"

In [63]:
right_join_full = pd.merge(students, courses, on='course_id', how='right', indicator=True)
right_join_full

,student_id,name,course_id,course_name,_merge
0,1.0,Anan,101,Math,both
1,2.0,Bella,102,Physics,both
2,3.0,Chai,103,Chemistry,both
3,NaN,NaN,105,Biology,right_only


In [64]:
right_only = right_join_full[right_join_full['_merge'] == 'right_only']
right_only

,student_id,name,course_id,course_name,_merge
3,NaN,NaN,105,Biology,right_only
